# Модуль 2 — контур гари и степень поражения (BS)

Вход: BS-чип 512x512 @ 20 м — Sentinel-2 pre/post (10 каналов: B2,B3,B4,B5,B6,B7,B8A,B11,B12,SCL), Sentinel-1 pre/post (VV,VH), AUX (dem, slope, landcover).

Выход: маска 0-3 (0 = не горело, 1/2/3 = слабая/средняя/сильная степень поражения).

Метрика: `IoU_burn` (бинарный контур гари, вес 0.35) + `mIoU_sev` (среднее IoU по классам 1/2/3, вес 0.30).

Модель: U-Net (обучается с нуля, вход — 33-канальный стек Sentinel-2/1 + спектральные индексы + AUX + облачная маска), обучение на Colab/Kaggle с GPU.

## 0. Настройка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Ноутбук самодостаточный — код `src/dataio.py`, `src/rle.py`, `src/metrics.py` встроен прямо сюда (см. следующие 3 ячейки), поэтому запуск в Colab работает при загрузке одного `.ipynb`-файла, без остального репозитория рядом.

In [ ]:
# === встроено из src/dataio.py - не редактировать здесь, правьте src/dataio.py и пересоберите ноутбук через build_notebooks.py ===
"""Chip readers for the КосмоХакатон fire-monitoring dataset.

Disk on the dev machine is essentially full, so `TarChipSource` reads every
GeoTIFF straight out of `fire-train-renamed.tar` into memory (via
rasterio.io.MemoryFile) and never extracts anything to disk. `DirChipSource`
is the same interface for a plain extracted folder, for when the organizers'
test set turns out to ship as loose files instead of a tar - swap the source,
keep every notebook/inference cell unchanged.

Band names are read from each GeoTIFF's own band descriptions (they're
embedded in the files), not hardcoded from the case statement PDF, so this
keeps working even if a channel gets reordered.
"""
from __future__ import annotations

import io
import tarfile
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile


class _ChipSourceBase:
    def _read_bytes(self, relpath: str) -> bytes:
        raise NotImplementedError

    def close(self) -> None:
        """No-op by default; TarChipSource overrides this to close its tarfile handle."""

    def __enter__(self):
        return self

    def __exit__(self, *exc) -> None:
        self.close()

    def read_named(self, relpath: str) -> dict:
        """Multi-band GeoTIFF -> dict keyed by band description, plus '_bands' (C,H,W),
        '_names' (tuple) and '_profile' (rasterio profile, has crs/transform/nodata)."""
        raw = self._read_bytes(relpath)
        with MemoryFile(raw) as mem, mem.open() as src:
            names = src.descriptions
            arr = src.read()
            profile = src.profile
        out = {"_bands": arr, "_names": tuple(names), "_profile": profile}
        for i, name in enumerate(names):
            if name:
                out[name] = arr[i]
        return out

    def read_csv(self, relpath: str) -> pd.DataFrame:
        return pd.read_csv(io.BytesIO(self._read_bytes(relpath)))

    def meta(self, kind: str) -> pd.DataFrame:
        return self.read_csv(f"{kind}/meta.csv")

    def chip_ids(self, kind: str) -> list[str]:
        return self.meta(kind)["chip_id"].tolist()

    def _try_load_mask(self, relpath: str) -> tuple[np.ndarray | None, float | None]:
        """Test chips ship with no mask (that's what we predict) - returns (None, None)
        instead of raising, so load_af/load_bs work unchanged on train, val AND test."""
        try:
            mask_raw = self.read_named(relpath)
        except FileNotFoundError:
            return None, None
        return mask_raw["_bands"][0], mask_raw["_profile"].get("nodata")

    def load_af(self, chip_id: str) -> dict:
        viirs = self.read_named(f"af/viirs/{chip_id}_VIIRS_I1-I5.tif")
        aux = self.read_named(f"af/aux/{chip_id}_AUX.tif")
        mask, mask_nodata = self._try_load_mask(f"af/masks/{chip_id}_MASK.tif")
        return {
            "chip_id": chip_id,
            "viirs": viirs,
            "aux": aux,
            "mask": mask,
            "mask_nodata": mask_nodata,
        }

    def load_bs(self, chip_id: str) -> dict:
        s2_pre = self.read_named(f"bs/sentinel2_pre/{chip_id}_Sentinel-2_pre.tif")
        s2_post = self.read_named(f"bs/sentinel2_post/{chip_id}_Sentinel-2_post.tif")
        s1_pre = self.read_named(f"bs/sentinel1_pre/{chip_id}_Sentinel-1_pre.tif")
        s1_post = self.read_named(f"bs/sentinel1_post/{chip_id}_Sentinel-1_post.tif")
        aux = self.read_named(f"bs/aux/{chip_id}_AUX.tif")
        mask, mask_nodata = self._try_load_mask(f"bs/masks/{chip_id}_MASK.tif")
        return {
            "chip_id": chip_id,
            "s2_pre": s2_pre,
            "s2_post": s2_post,
            "s1_pre": s1_pre,
            "s1_post": s1_post,
            "aux": aux,
            "mask": mask,
            "mask_nodata": mask_nodata,
        }


class TarChipSource(_ChipSourceBase):
    """Reads chips directly out of fire-train-renamed.tar, no extraction to disk."""

    def __init__(self, tar_path: str | Path, root: str = "train"):
        self.tar_path = str(tar_path)
        self.root = root
        self._tar = tarfile.open(self.tar_path)

    def _read_bytes(self, relpath: str) -> bytes:
        member = f"{self.root}/{relpath}"
        try:
            buf = self._tar.extractfile(member)
        except KeyError:
            # tarfile raises KeyError (not FileNotFoundError) for a missing member -
            # normalize it so callers (e.g. _try_load_mask) can catch one exception
            # type regardless of which ChipSource they're using.
            raise FileNotFoundError(member) from None
        if buf is None:
            raise FileNotFoundError(member)
        return buf.read()

    def close(self) -> None:
        self._tar.close()


class DirChipSource(_ChipSourceBase):
    """Same interface as TarChipSource, for an already-extracted directory
    (e.g. the organizers' --data-dir at inference time)."""

    def __init__(self, root_dir: str | Path):
        self.root_dir = Path(root_dir)

    def _read_bytes(self, relpath: str) -> bytes:
        path = self.root_dir / relpath
        return path.read_bytes()


def nodata_to_ignore(mask: np.ndarray, nodata: float | None) -> np.ndarray:
    """Boolean valid-pixel mask (True = usable) from a mask array + its nodata value."""
    if nodata is None:
        return np.ones_like(mask, dtype=bool)
    return mask != nodata


def iter_chips(source: _ChipSourceBase, kind: str, chip_ids: Iterable[str] | None = None):
    """Yield load_af()/load_bs() dicts for every chip_id (or all chips if None)."""
    loader = source.load_af if kind == "af" else source.load_bs
    ids = list(chip_ids) if chip_ids is not None else source.chip_ids(kind)
    for chip_id in ids:
        yield loader(chip_id)


In [ ]:
# === встроено из src/rle.py - не редактировать здесь, правьте src/rle.py и пересоберите ноутбук через build_notebooks.py ===
"""RLE encode/decode matching the КосмоХакатон submission.csv spec.

Pixel numbering is row-major (left-to-right, top-to-bottom), 1-indexed.
Runs are reported in ascending start order and never overlap or touch,
which is exactly what run-length-encoding a boolean mask gives you for free.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def encode_rle(mask: np.ndarray) -> str:
    """2D boolean/0-1 array -> RLE string ("start len start len ..."), '' if empty."""
    flat = np.asarray(mask, dtype=bool).ravel(order="C")
    if not flat.any():
        return ""
    padded = np.concatenate(([0], flat.astype(np.int8), [0]))
    diffs = np.diff(padded)
    starts_0idx = np.flatnonzero(diffs == 1)
    ends_k = np.flatnonzero(diffs == -1)
    lengths = ends_k - starts_0idx
    starts_1idx = starts_0idx + 1
    return " ".join(f"{s} {l}" for s, l in zip(starts_1idx, lengths))


def decode_rle(rle, shape: tuple[int, int]) -> np.ndarray:
    """RLE string + (H, W) -> boolean mask.

    An empty mask round-trips through CSV as an empty field, which pandas reads back
    as NaN (float) rather than "" - this is also how the organizers' own
    sample_submission.csv represents "no pixels of this class", so NaN/None are treated
    the same as an empty string here rather than as errors.
    """
    mask = np.zeros(shape[0] * shape[1], dtype=bool)
    if rle is None or (isinstance(rle, float) and np.isnan(rle)):
        rle = ""
    rle = str(rle).strip()
    if rle:
        nums = list(map(int, rle.split()))
        for start, length in zip(nums[0::2], nums[1::2]):
            s0 = start - 1
            mask[s0 : s0 + length] = True
    return mask.reshape(shape, order="C")


def submission_row(chip_id: str, class_id: int, mask: np.ndarray) -> dict:
    return {"chip_id": chip_id, "class_id": int(class_id), "rle": encode_rle(mask)}


def validate_submission(df: pd.DataFrame, sample_submission: pd.DataFrame) -> list[str]:
    """Structural checks against the shipped sample_submission.csv. Empty list == OK."""
    problems = []
    required = set(zip(sample_submission["chip_id"], sample_submission["class_id"]))
    got = set(zip(df["chip_id"], df["class_id"]))
    missing = required - got
    extra = got - required
    if missing:
        problems.append(f"missing {len(missing)} (chip_id, class_id) pairs, e.g. {list(missing)[:5]}")
    if extra:
        problems.append(f"unexpected {len(extra)} (chip_id, class_id) pairs, e.g. {list(extra)[:5]}")
    # NaN in "rle" is NOT an error: an empty mask round-trips through to_csv/read_csv as
    # an empty field, which pandas parses as NaN - decode_rle() treats it as "" on purpose,
    # matching how the organizers' own sample_submission.csv represents an empty mask.
    if df.duplicated(subset=["chip_id", "class_id"]).any():
        problems.append("duplicate (chip_id, class_id) rows found")
    return problems


def validate_submission_full(
    df: pd.DataFrame,
    sample_submission: pd.DataFrame,
    chip_shapes: dict[str, tuple[int, int]],
) -> list[str]:
    """Structural checks plus per-chip overlap/out-of-bounds decoding checks."""
    problems = validate_submission(df, sample_submission)
    for chip_id, group in df.groupby("chip_id"):
        shape = chip_shapes.get(chip_id)
        if shape is None:
            continue
        total = np.zeros(shape, dtype=np.uint8)
        for _, row in group.iterrows():
            try:
                m = decode_rle(row["rle"], shape)
            except Exception as e:  # noqa: BLE001 - want the chip_id in the message
                problems.append(f"{chip_id}/{row['class_id']}: RLE decode failed ({e})")
                continue
            total += m.astype(np.uint8)
        if (total > 1).any():
            problems.append(f"{chip_id}: overlapping masks between classes")
    return problems


In [ ]:
# === встроено из src/metrics.py - не редактировать здесь, правьте src/metrics.py и пересоберите ноутбук через build_notebooks.py ===
"""Competition score, computed exactly like the organizers compute it:
pixels from every chip are pooled into one confusion matrix per class
(micro-averaging) *before* precision/recall/F1/IoU are derived - not
averaged per-chip and then across chips.

Score = 0.35 * F1_af + 0.35 * IoU_burn + 0.30 * mIoU_sev
"""
from __future__ import annotations

import numpy as np


class ConfusionAccumulator:
    """Pools TP/FP/FN across an arbitrary number of update() calls (= chips)."""

    def __init__(self, classes: tuple[int, ...]):
        self.classes = tuple(classes)
        self.tp = {c: 0 for c in self.classes}
        self.fp = {c: 0 for c in self.classes}
        self.fn = {c: 0 for c in self.classes}

    def update(self, pred: np.ndarray, true: np.ndarray, valid: np.ndarray | None = None) -> None:
        if valid is None:
            valid = np.ones_like(true, dtype=bool)
        for c in self.classes:
            p = (pred == c) & valid
            t = (true == c) & valid
            self.tp[c] += int(np.sum(p & t))
            self.fp[c] += int(np.sum(p & ~t))
            self.fn[c] += int(np.sum(~p & t))

    def iou(self, c: int) -> float:
        tp, fp, fn = self.tp[c], self.fp[c], self.fn[c]
        denom = tp + fp + fn
        return 1.0 if denom == 0 else tp / denom

    def precision_recall_f1(self, c: int) -> tuple[float, float, float]:
        tp, fp, fn = self.tp[c], self.fp[c], self.fn[c]
        if tp + fp == 0 and tp + fn == 0:
            return 1.0, 1.0, 1.0
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        return precision, recall, f1


def binarize_burn(mask: np.ndarray) -> np.ndarray:
    """BS mask (0/1/2/3) -> binary burned/not-burned mask, for IoU_burn."""
    return (mask >= 1).astype(np.uint8)


def f1_af(acc: ConfusionAccumulator) -> float:
    return acc.precision_recall_f1(1)[2]


def iou_burn(acc_binary: ConfusionAccumulator) -> float:
    """acc_binary must have been fed binarize_burn() masks with classes=(1,)."""
    return acc_binary.iou(1)


def miou_severity(acc_sev: ConfusionAccumulator) -> float:
    """acc_sev must have been fed the raw 0..3 masks with classes=(1, 2, 3)."""
    return sum(acc_sev.iou(c) for c in (1, 2, 3)) / 3


def score(f1_af_value: float, iou_burn_value: float, miou_sev_value: float) -> float:
    return 0.35 * f1_af_value + 0.35 * iou_burn_value + 0.30 * miou_sev_value


In [ ]:
# === встроено из src/bs_pipeline.py - не редактировать здесь, правьте src/bs_pipeline.py и пересоберите ноутбук через build_notebooks.py ===
"""BS module (burn contour + severity) inference pipeline: feature stacks,
siamese U-Net ensemble, TTA, thresholds, postprocessing.

Same logic as embedded in notebooks/02_bs_burn_severity.ipynb (via
build_notebooks.py) - kept here too as a plain importable module so
`inference.py` can run outside Colab/Jupyter using just a local directory of
weights. If you change feature engineering or model architecture here, mirror
the change in build_notebooks.py (and vice versa) - the two are not auto-synced.

Only inference-time code lives here. Training-only code (Dataset/augmentation,
the training loop, class weights, threshold search) stays in the notebook,
since training only ever happens on Colab.
"""
from __future__ import annotations

import json
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage as ndi

BS_S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8A", "B11", "B12"]
SCL_CLOUD_CLASSES = {3, 8, 9, 10}  # shadow / cloud medium+high probability / cirrus

BS_PRE_CHANNELS = BS_S2_BANDS + ["NDVI", "NBR", "NDMI", "MIRBI", "BAI", "GEMI", "VV_db", "VH_db", "RVI"]
BS_STATIC_CHANNELS = [
    "dNBR", "RdNBR", "RBR", "dVV", "dVH", "RBD", "LogRBR", "dRVI",
    "dem", "slope", "landcover", "cloud_mask",
]


def _refl(band):
    """Sentinel-2 L2A stores reflectance as uint16 * 10000."""
    return band.astype(np.float32) / 10000.0


def _ndvi(bands):
    red, nir = bands["B4"].astype(np.float32), bands["B8A"].astype(np.float32)
    return (nir - red) / (nir + red + 1e-6)


def _nbr(bands):
    b8a, b12 = bands["B8A"].astype(np.float32), bands["B12"].astype(np.float32)
    return (b8a - b12) / (b8a + b12 + 1e-6)


def _ndmi(bands):
    nir, swir1 = bands["B8A"].astype(np.float32), bands["B11"].astype(np.float32)
    return (nir - swir1) / (nir + swir1 + 1e-6)


def _mirbi(bands):
    swir1, swir2 = _refl(bands["B11"]), _refl(bands["B12"])
    return 10.0 * swir2 - 9.8 * swir1 + 2.0


def _bai(bands):
    red, nir = _refl(bands["B4"]), _refl(bands["B8A"])
    val = 1.0 / ((0.1 - red) ** 2 + (0.06 - nir) ** 2 + 1e-6)
    return np.clip(val, 0, 50)


def _gemi(bands):
    red, nir = _refl(bands["B4"]), _refl(bands["B8A"])
    eta = (2.0 * (nir ** 2 - red ** 2) + 1.5 * nir + 0.5 * red) / (nir + red + 0.5 + 1e-6)
    return eta * (1 - 0.25 * eta) - (red - 0.125) / (1 - red + 1e-6)


def _s1_db(band, lo=-30.0, hi=5.0):
    """int16 stores dB*100 (confirmed on real data) - convert to dB and clip to a
    physically sane range (radar shadow/layover outliers otherwise dominate
    normalization statistics)."""
    return np.clip(band.astype(np.float32) / 100.0, lo, hi)


def _rvi(vv_db, vh_db):
    """Radar Vegetation Index - defined for LINEAR power, not dB."""
    vv_lin, vh_lin = 10.0 ** (vv_db / 10.0), 10.0 ** (vh_db / 10.0)
    return 4.0 * vh_lin / (vv_lin + vh_lin + 1e-6)


def _build_date_branch(bands, s1):
    vv_db, vh_db = _s1_db(s1["VV"]), _s1_db(s1["VH"])
    stack = [bands[b].astype(np.float32) for b in BS_S2_BANDS] + [
        _ndvi(bands), _nbr(bands), _ndmi(bands), _mirbi(bands), _bai(bands), _gemi(bands),
        vv_db, vh_db, _rvi(vv_db, vh_db),
    ]
    return np.stack(stack, axis=0).astype(np.float32)


def build_bs_feature_stack_siamese(chip):
    """(pre_stack, post_stack, static_stack), shapes (18,H,W), (18,H,W), (12,H,W)."""
    s2pre, s2post = chip["s2_pre"], chip["s2_post"]
    s1pre, s1post = chip["s1_pre"], chip["s1_post"]
    aux = chip["aux"]

    pre_stack = _build_date_branch(s2pre, s1pre)
    post_stack = _build_date_branch(s2post, s1post)

    nbr_pre, nbr_post = _nbr(s2pre), _nbr(s2post)
    dnbr = nbr_pre - nbr_post
    rdnbr = np.clip(dnbr / (np.sqrt(np.abs(nbr_pre) / 1000.0) + 1e-6), -1000, 1000)
    rbr = np.clip(dnbr / (nbr_pre + 1.001), -3, 3)

    vv_pre_db, vh_pre_db = _s1_db(s1pre["VV"]), _s1_db(s1pre["VH"])
    vv_post_db, vh_post_db = _s1_db(s1post["VV"]), _s1_db(s1post["VH"])
    dvv = np.clip(vv_post_db - vv_pre_db, -20, 20)
    dvh = np.clip(vh_post_db - vh_pre_db, -20, 20)
    rbd = np.clip(vh_pre_db - vh_post_db, -20, 20)
    vh_pre_lin, vh_post_lin = 10.0 ** (vh_pre_db / 10.0), 10.0 ** (vh_post_db / 10.0)
    log_rbr = np.clip(np.log10((vh_pre_lin + 1e-6) / (vh_post_lin + 1e-6)), -3, 3)
    drvi = _rvi(vv_post_db, vh_post_db) - _rvi(vv_pre_db, vh_pre_db)

    cloud_mask = np.isin(s2post["SCL"], list(SCL_CLOUD_CLASSES)).astype(np.float32)
    static_stack = np.stack([
        dnbr, rdnbr, rbr,
        dvv, dvh, rbd, log_rbr, drvi,
        aux["dem"].astype(np.float32), aux["slope"].astype(np.float32),
        aux["landcover"].astype(np.float32), cloud_mask,
    ], axis=0).astype(np.float32)

    clean = lambda a: np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return clean(pre_stack), clean(post_stack), clean(static_stack)


# ---------------------------------------------------------------------------
# Per-pixel feature stack for the LightGBM severity model.
#
# Why a per-pixel GBM at all, next to the siamese U-Net: burn severity is
# largely a per-pixel spectral problem (dNBR conditioned on land cover - the
# case statement itself says fixed dNBR scales fail because forest/steppe/
# cropland need different calibration). A GBM sees ~47M labelled pixels from
# 179 chips, while the U-Net sees 179 images - so the GBM is far less
# data-starved on exactly the part of Score that has the most headroom
# (mIoU_sev), and it is an order of magnitude cheaper at inference.
# ---------------------------------------------------------------------------

BS_PIXEL_FEATURE_NAMES = (
    [f"{b}_pre" for b in BS_S2_BANDS] + [f"{b}_post" for b in BS_S2_BANDS] + [
        "NDVI_pre", "NDVI_post", "dNDVI",
        "NBR_pre", "NBR_post", "dNBR", "RdNBR", "RBR",
        "NDMI_pre", "NDMI_post", "dNDMI",
        "MIRBI_pre", "MIRBI_post", "dMIRBI",
        "BAI_pre", "BAI_post", "dBAI",
        "GEMI_pre", "GEMI_post", "dGEMI",
        "VV_pre_db", "VH_pre_db", "VV_post_db", "VH_post_db",
        "dVV", "dVH", "RBD", "LogRBR",
        "RVI_pre", "RVI_post", "dRVI",
        "dem", "slope", "landcover", "cloud_mask",
        "dNBR_mean5", "dNBR_mean15", "dNBR_local_contrast",
        # --- scene-relative + texture block ---------------------------------
        # The case statement itself says a single dNBR scale does not transfer
        # between forest/steppe/cropland. Absolute dNBR is therefore a weak
        # feature across scenes; its rank/z-score WITHIN the chip is not, and
        # local std separates a real burn edge from sensor noise. Measured on
        # validation: IoU_burn 0.5880 -> 0.5891, mIoU_sev 0.5786 -> 0.5847,
        # with class-1 (light burn) IoU 0.376 -> 0.387.
        "dNBR_z", "dNBR_rank", "dNBR_med_rel",
        "NBR_post_z", "NBR_post_rank",
        "NDVI_post_z", "dNDVI_rank",
        "MIRBI_post_z", "dNBR_std5", "dNBR_std15", "dNBR_mean31",
        "NBR_post_mean5", "dNDVI_mean5", "dNBR_rank_mean15",
    ]
)
BS_PIXEL_LANDCOVER_IDX = BS_PIXEL_FEATURE_NAMES.index("landcover")


def _chip_z(x):
    """z-score within the chip - makes a spectral index scene-relative."""
    return (x - float(np.nanmean(x))) / (float(np.nanstd(x)) + 1e-6)


def _chip_rank(x):
    """percentile rank within the chip, 0..1 - fully scale-invariant."""
    flat = x.ravel()
    order = np.argsort(flat, kind="stable")
    r = np.empty(flat.shape[0], np.float32)
    r[order] = np.arange(flat.shape[0], dtype=np.float32) / max(flat.shape[0] - 1, 1)
    return r.reshape(x.shape)


def _std_filter(x, size):
    from scipy.ndimage import uniform_filter

    m = uniform_filter(x, size=size)
    return np.sqrt(np.maximum(uniform_filter(x * x, size=size) - m * m, 0.0))


def build_bs_pixel_features(chip):
    """(C, H, W) float32 in BS_PIXEL_FEATURE_NAMES order, for the LightGBM model.

    `dNBR_mean5`/`dNBR_mean15`/`dNBR_local_contrast` give the otherwise purely
    per-pixel model a bit of spatial context for free (a burnt pixel sits inside
    a burnt neighbourhood; an isolated low-dNBR pixel is usually noise)."""
    from scipy.ndimage import uniform_filter

    s2pre, s2post = chip["s2_pre"], chip["s2_post"]
    s1pre, s1post = chip["s1_pre"], chip["s1_post"]
    aux = chip["aux"]

    ndvi_pre, ndvi_post = _ndvi(s2pre), _ndvi(s2post)
    nbr_pre, nbr_post = _nbr(s2pre), _nbr(s2post)
    dnbr = nbr_pre - nbr_post
    rdnbr = np.clip(dnbr / (np.sqrt(np.abs(nbr_pre) / 1000.0) + 1e-6), -1000, 1000)
    rbr = np.clip(dnbr / (nbr_pre + 1.001), -3, 3)
    ndmi_pre, ndmi_post = _ndmi(s2pre), _ndmi(s2post)
    mirbi_pre, mirbi_post = _mirbi(s2pre), _mirbi(s2post)
    bai_pre, bai_post = _bai(s2pre), _bai(s2post)
    gemi_pre, gemi_post = _gemi(s2pre), _gemi(s2post)

    vv_pre_db, vh_pre_db = _s1_db(s1pre["VV"]), _s1_db(s1pre["VH"])
    vv_post_db, vh_post_db = _s1_db(s1post["VV"]), _s1_db(s1post["VH"])
    dvv = np.clip(vv_post_db - vv_pre_db, -20, 20)
    dvh = np.clip(vh_post_db - vh_pre_db, -20, 20)
    rbd = np.clip(vh_pre_db - vh_post_db, -20, 20)
    vh_pre_lin, vh_post_lin = 10.0 ** (vh_pre_db / 10.0), 10.0 ** (vh_post_db / 10.0)
    log_rbr = np.clip(np.log10((vh_pre_lin + 1e-6) / (vh_post_lin + 1e-6)), -3, 3)
    rvi_pre, rvi_post = _rvi(vv_pre_db, vh_pre_db), _rvi(vv_post_db, vh_post_db)

    dnbr_mean5 = uniform_filter(dnbr, size=5)
    dnbr_mean15 = uniform_filter(dnbr, size=15)

    dndvi = ndvi_post - ndvi_pre
    dnbr_rank = _chip_rank(dnbr)

    cloud_mask = np.isin(s2post["SCL"], list(SCL_CLOUD_CLASSES)).astype(np.float32)

    feats = np.stack(
        [s2pre[b].astype(np.float32) for b in BS_S2_BANDS]
        + [s2post[b].astype(np.float32) for b in BS_S2_BANDS]
        + [
            ndvi_pre, ndvi_post, ndvi_post - ndvi_pre,
            nbr_pre, nbr_post, dnbr, rdnbr, rbr,
            ndmi_pre, ndmi_post, ndmi_post - ndmi_pre,
            mirbi_pre, mirbi_post, mirbi_post - mirbi_pre,
            bai_pre, bai_post, bai_post - bai_pre,
            gemi_pre, gemi_post, gemi_post - gemi_pre,
            vv_pre_db, vh_pre_db, vv_post_db, vh_post_db,
            dvv, dvh, rbd, log_rbr,
            rvi_pre, rvi_post, rvi_post - rvi_pre,
            aux["dem"].astype(np.float32), aux["slope"].astype(np.float32),
            aux["landcover"].astype(np.float32), cloud_mask,
            dnbr_mean5, dnbr_mean15, dnbr - dnbr_mean15,
            _chip_z(dnbr), dnbr_rank, dnbr - float(np.nanmedian(dnbr)),
            _chip_z(nbr_post), _chip_rank(nbr_post),
            _chip_z(ndvi_post), _chip_rank(dndvi),
            _chip_z(mirbi_post), _std_filter(dnbr, 5), _std_filter(dnbr, 15),
            uniform_filter(dnbr, size=31),
            uniform_filter(nbr_post, size=5), uniform_filter(dndvi, size=5),
            uniform_filter(dnbr_rank, size=15),
        ],
        axis=0,
    ).astype(np.float32)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


# LightGBM stops walking the remaining trees for a pixel once the leading class
# is ahead by this margin. On a 512x512 chip almost every pixel is obviously
# unburnt, so this is where the time goes and where it can be given back:
# the model is 181 iterations x 4 classes = 724 trees over 262144 pixels.
#
# The margin was chosen by running the WHOLE BS pipeline (LightGBM -> refiner ->
# threshold -> postprocessing) on the validation split and scoring it with the
# organizers' own metric, not by eyeballing probabilities:
#
#   margin off -> 2.29 s/chip           IoU_burn 0.71323  mIoU_sev 0.67410
#   margin 7.0 -> 1.85 s/chip (1.24x)   IoU_burn 0.71323  mIoU_sev 0.67410
#   margin 5.0 -> 1.50 s/chip (1.53x)   IoU_burn 0.71339  mIoU_sev 0.67428
#   margin 3.5 -> 1.17 s/chip (1.95x)   IoU_burn 0.71320  mIoU_sev 0.67418
#
# At 3.5 the module's contribution to Score moves by +0.00001 - noise. (Pixel
# level, 0.038% of pixels across the test set do flip, but they are pixels the
# threshold was never going to be decided by, which is exactly why the metric
# does not move.)
#
# This matters for grading: the BS module stays CPU-bound on LightGBM even on
# the organizers' H100, because the GPU only removes the refiner. Halving the
# LightGBM time is what puts the whole inference inside the "under 5 minutes"
# bracket instead of on its boundary.
BS_LGB_EARLY_STOP_MARGIN = 3.5
BS_LGB_EARLY_STOP_FREQ = 10


def _lgb_predict(model, flat):
    """Predict with early stopping, falling back if the installed LightGBM does
    not accept those keywords - a speed optimisation must never be the reason
    inference fails on someone else's machine."""
    try:
        return model.predict(
            flat, num_iteration=model.best_iteration, pred_early_stop=True,
            pred_early_stop_freq=BS_LGB_EARLY_STOP_FREQ,
            pred_early_stop_margin=BS_LGB_EARLY_STOP_MARGIN)
    except (TypeError, ValueError):
        return model.predict(flat, num_iteration=model.best_iteration)


def predict_bs_proba_lgb(chip, model):
    """Full-image 4-class probabilities from the LightGBM model -> (4, H, W)."""
    feats = build_bs_pixel_features(chip)
    c, h, w = feats.shape
    probs = _lgb_predict(model, feats.reshape(c, -1).T)
    return probs.T.reshape(4, h, w).astype(np.float32)


# --------------------------------------------------------------------------
# Stacked spatial refiner
#
# The per-pixel LightGBM saturates around IoU_burn 0.59 / mIoU_sev 0.585 on
# validation, and the class it fails on is class 1 (light burn, IoU 0.39):
# spectrally light burn sits between unburnt and moderate burn, so a model that
# sees one pixel at a time cannot separate them. What does separate them is
# context - light burn forms rings around heavier burn, and fire scars are
# contiguous shapes. Hence a CNN that takes the LightGBM probability maps as
# input and refines them spatially, rather than replacing them.
#
# Two design choices make this safe rather than a gamble:
#   * ImageNet-pretrained ResNet34 encoder - 224 training chips is far too few
#     to learn low-level filters from scratch (the earlier from-scratch siamese
#     U-Net scored 0.274 against LightGBM's 0.374, i.e. it was data-starved);
#   * a residual connection onto log(P_lgb) with a ZERO-initialised output
#     layer, so at step 0 the network reproduces the LightGBM probabilities
#     exactly and training can only move away from them if that lowers the loss.
# --------------------------------------------------------------------------
BS_REFINER_FEATURE_NAMES = [
    "dNBR", "dNBR_rank", "dNBR_z", "dNBR_std5", "NBR_pre", "NBR_post",
    "dNDVI", "NDVI_post", "dNDMI", "dMIRBI", "dVV", "dVH",
    "dem", "slope", "landcover", "cloud_mask",
]
BS_REFINER_IDX = [BS_PIXEL_FEATURE_NAMES.index(n) for n in BS_REFINER_FEATURE_NAMES]
BS_REFINER_IN_CH = 4 + len(BS_REFINER_FEATURE_NAMES)


def build_refiner_input(pixel_feats, lgb_proba):
    """(BS_REFINER_IN_CH, H, W) float32: 4 LightGBM probability maps + a compact
    raw block, each raw channel z-scored WITHIN the chip so no normalisation
    statistics have to be stored alongside the weights."""
    raw = pixel_feats[BS_REFINER_IDX].astype(np.float32)
    out = np.empty((BS_REFINER_IN_CH, raw.shape[1], raw.shape[2]), np.float32)
    out[:4] = lgb_proba
    for i, name in enumerate(BS_REFINER_FEATURE_NAMES):
        if name == "landcover":
            out[4 + i] = raw[i] / 100.0            # categorical codes 10..100
        elif name == "cloud_mask":
            out[4 + i] = raw[i]                     # already 0/1
        else:
            out[4 + i] = _chip_z(raw[i])
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


class _Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = ConvBlock(cin + cskip, cout)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class BSRefiner(nn.Module):
    """ResNet34-U-Net that refines LightGBM probabilities. `pretrained=False` at
    inference time so constructing it never touches the network - the encoder
    weights come from the saved checkpoint like everything else."""

    def __init__(self, in_ch=BS_REFINER_IN_CH, n_classes=4, pretrained=False):
        super().__init__()
        import torchvision

        enc = torchvision.models.resnet34(
            weights="IMAGENET1K_V1" if pretrained else None)
        w = enc.conv1.weight.data.clone()
        enc.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            reps = (in_ch + 2) // 3
            enc.conv1.weight.data = w.repeat(1, reps, 1, 1)[:, :in_ch] * (3.0 / in_ch)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool, self.l1, self.l2 = enc.maxpool, enc.layer1, enc.layer2
        self.l3, self.l4 = enc.layer3, enc.layer4
        self.d4, self.d3 = _Up(512, 256, 256), _Up(256, 128, 128)
        self.d2, self.d1 = _Up(128, 64, 64), _Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_classes, 1),
        )
        # identity start: logits = 0 + log(P_lgb)  =>  softmax == P_lgb
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x):
        lgb_logit = torch.log(x[:, :4].clamp_min(1e-6))
        s0 = self.stem(x)
        x1 = self.l1(self.pool(s0))
        x2, x3 = self.l2(x1), None
        x3 = self.l3(x2)
        x4 = self.l4(x3)
        y = self.d1(self.d2(self.d3(self.d4(x4, x3), x2), x1), s0)
        y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(y) + lgb_logit


def save_refiner_state(model, path):
    """Checkpoints go to disk in fp16: 98.5 MB -> 49.3 MB, which keeps them well
    under GitHub's hard 100 MB per-file limit. Measured cost of the round-trip:
    max |delta softmax| = 3.5e-6 and not a single pixel changes its argmax."""
    torch.save({k: (v.half() if v.is_floating_point() else v)
                for k, v in model.state_dict().items()}, path)


def load_refiner_state(model, path, device="cpu"):
    """Accepts both fp16 and fp32 checkpoints - always casts to the model's dtype."""
    sd = torch.load(path, map_location=device)
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v)
                            for k, v in sd.items()})
    return model


@torch.no_grad()
def predict_bs_proba_refiner(refiner_input, models, device="cpu", use_tta=True):
    """Average softmax over the refiner ensemble (and flip-TTA) -> (4, H, W)."""
    x = torch.from_numpy(refiner_input).unsqueeze(0).to(device)
    variants = [(x, None)]
    if use_tta:
        variants += [(torch.flip(x, [-1]), [-1]), (torch.flip(x, [-2]), [-2]),
                     (torch.flip(x, [-1, -2]), [-1, -2])]
    total, n = 0.0, 0
    for model in models:
        for xv, dims in variants:
            p = torch.softmax(model(xv), dim=1)
            if dims is not None:
                p = torch.flip(p, dims)
            total = total + p
            n += 1
    return (total / n)[0].float().cpu().numpy()


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SharedEncoder(nn.Module):
    def __init__(self, in_channels, base=32):
        super().__init__()
        self.input_norm = nn.BatchNorm2d(in_channels, affine=False)
        self.enc1 = ConvBlock(in_channels, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 8, base * 16)

    def forward(self, x):
        x = self.input_norm(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        return e1, e2, e3, e4, b


class SiameseUNet(nn.Module):
    """Shared-weight encoder for pre/post (change-invariant features, xView2
    1st/2nd place and BAM-CD/FLOGA pattern) + a static/diff stream fused once
    at the bottleneck."""

    def __init__(self, pair_channels, static_channels, n_classes, base=32):
        super().__init__()
        self.encoder = SharedEncoder(pair_channels, base)
        self.static_norm = nn.BatchNorm2d(static_channels, affine=False)
        self.static_stem = ConvBlock(static_channels, base * 16)
        self.bottleneck_fuse = nn.Conv2d(base * 16 * 3, base * 16, 1)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 8 * 3, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 4 * 3, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 2 * 3, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 3, base)
        self.out_conv = nn.Conv2d(base, n_classes, 1)

    def forward(self, pre, post, static):
        e1p, e2p, e3p, e4p, bp = self.encoder(pre)
        e1o, e2o, e3o, e4o, bo = self.encoder(post)
        static_feat = F.adaptive_avg_pool2d(self.static_stem(self.static_norm(static)), bp.shape[-2:])
        b = self.bottleneck_fuse(torch.cat([bp, bo, static_feat], dim=1))
        d4 = self.dec4(torch.cat([self.up4(b), e4p, e4o], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3p, e3o], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2p, e2o], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1p, e1o], dim=1))
        return self.out_conv(d1)


def _tta_variants(pre, post, static):
    variants = []
    for flip_h, flip_v in [(False, False), (True, False), (False, True), (True, True)]:
        p, o, s = pre, post, static
        if flip_h:
            p, o, s = torch.flip(p, dims=[-1]), torch.flip(o, dims=[-1]), torch.flip(s, dims=[-1])
        if flip_v:
            p, o, s = torch.flip(p, dims=[-2]), torch.flip(o, dims=[-2]), torch.flip(s, dims=[-2])
        variants.append((p, o, s, flip_h, flip_v))
    return variants


def predict_bs_proba_tta(pre, post, static, models, device="cpu", use_tta=True):
    """Average softmax across the model ensemble and (optionally) TTA variants.
    pre/post/static are (C,H,W) tensors with no batch dim. Returns (4,H,W) numpy."""
    pre_b = pre.unsqueeze(0).to(device)
    post_b = post.unsqueeze(0).to(device)
    static_b = static.unsqueeze(0).to(device)
    variants = _tta_variants(pre_b, post_b, static_b) if use_tta else [(pre_b, post_b, static_b, False, False)]
    total = None
    with torch.no_grad():
        for model in models:
            for p, o, s, flip_h, flip_v in variants:
                probs = F.softmax(model(p, o, s), dim=1)
                if flip_h:
                    probs = torch.flip(probs, dims=[-1])
                if flip_v:
                    probs = torch.flip(probs, dims=[-2])
                total = probs if total is None else total + probs
    n = len(models) * len(variants)
    return (total / n).squeeze(0).cpu().numpy()


def apply_bs_thresholds(probs, burn_threshold=0.5, severity_weights=(1.0, 1.0, 1.0)):
    """"Is there burn at all" (P(1)+P(2)+P(3)) is thresholded separately from the
    severity argmax among classes 1/2/3 - IoU_burn (weight 0.35) and mIoU_sev
    (weight 0.30) are optimized somewhat independently in Score.

    `severity_weights` rescale the three severity probabilities before the
    argmax. mIoU_sev is a MEAN of per-class IoU, so a rare class is worth as
    much as a common one and is worth biasing towards; the weights are searched
    on validation by coordinate ascent."""
    p_burn = probs[1] + probs[2] + probs[3]
    is_burn = p_burn >= burn_threshold
    w = np.asarray(severity_weights, dtype=np.float32).reshape(3, 1, 1)
    severity = 1 + np.argmax(probs[1:4] * w, axis=0)
    return np.where(is_burn, severity, 0).astype(np.uint8)


def postprocess_bs_mask(pred, min_blob_size=20, dilate_classes=(2, 3), dilate_iters=1):
    """Drop small isolated false-positive blobs + class-specific dilation for 2/3
    (xView2 1st place pattern: network boundaries for the damaged class tend to
    be under-called at the edges)."""
    out = pred.copy()
    labeled, n = ndi.label(pred > 0)
    for i in range(1, n + 1):
        blob = labeled == i
        if blob.sum() < min_blob_size:
            out[blob] = 0
    for c in dilate_classes:
        class_mask = out == c
        if class_mask.any():
            dilated = ndi.binary_dilation(class_mask, iterations=dilate_iters)
            out[dilated & (out == 0)] = c
    return out


def try_slic_smooth(pred, guide_rgb, n_segments=300):
    """Optional SLIC superpixel majority-vote smoothing; silently returns pred
    unchanged if scikit-image isn't installed - must never break the pipeline."""
    try:
        from skimage.segmentation import slic
    except ImportError:
        return pred
    segments = slic(guide_rgb, n_segments=n_segments, compactness=10, start_label=0, channel_axis=-1)
    out = pred.copy()
    for seg_id in np.unique(segments):
        seg_mask = segments == seg_id
        values, counts = np.unique(pred[seg_mask], return_counts=True)
        out[seg_mask] = values[np.argmax(counts)]
    return out


def predict_bs_mask(chip_id, source, lgb_model=None, unet_models=None, refiner_models=None,
                     strategy="lgb", burn_threshold=0.76, min_blob_size=35,
                     severity_weights=(1.0, 1.0, 1.0), refiner_alpha=1.0,
                     device="cpu", use_tta=True):
    """Full BS pipeline for one chip -> uint8 mask (0..3). RLE-encoding per class
    (`submission_row(chip_id, c, mask == c)` for c in 1,2,3) is the caller's job.

    `strategy` is whatever won on validation in the training notebook:
    'lgb'     - per-pixel LightGBM alone,
    'refiner' - LightGBM probabilities refined by the stacked ResNet34-U-Net
                (blended with the raw LightGBM maps by `refiner_alpha`),
    'unet'    - the legacy siamese U-Net ensemble,
    'blend'   - mean of LightGBM and the legacy siamese U-Net."""
    chip = source.load_bs(chip_id)

    probs_lgb = None
    if strategy in ("lgb", "blend", "refiner"):
        pixel_feats = build_bs_pixel_features(chip)
        c, h, w = pixel_feats.shape
        probs_lgb = _lgb_predict(
            lgb_model, pixel_feats.reshape(c, -1).T
        ).T.reshape(4, h, w).astype(np.float32)

    if strategy == "refiner":
        refined = predict_bs_proba_refiner(
            build_refiner_input(pixel_feats, probs_lgb), refiner_models,
            device=device, use_tta=use_tta,
        )
        probs = refiner_alpha * refined + (1.0 - refiner_alpha) * probs_lgb
    elif strategy == "lgb":
        probs = probs_lgb
    else:
        pre, post, static = build_bs_feature_stack_siamese(chip)
        probs_unet = predict_bs_proba_tta(
            torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(static),
            unet_models, device=device, use_tta=use_tta,
        )
        probs = probs_unet if strategy == "unet" else 0.5 * (probs_lgb + probs_unet)

    pred_mask = apply_bs_thresholds(probs, burn_threshold=burn_threshold,
                                     severity_weights=severity_weights)
    if min_blob_size:
        pred_mask = postprocess_bs_mask(pred_mask, min_blob_size=min_blob_size, dilate_classes=())
    return pred_mask


def save_bs_artifacts(weights_dir, models, burn_threshold, extra_meta=None):
    """Mirrors exactly what the training notebook writes per-model to
    CHECKPOINT_DIR - a weights folder saved this way can be copied straight into
    `weights/bs/` for `inference.py`."""
    os.makedirs(weights_dir, exist_ok=True)
    for i, m in enumerate(models):
        torch.save(m.state_dict(), os.path.join(weights_dir, f"bs_unet_{i}_best.pt"))
    meta = {
        "n_models": len(models),
        "pre_channels": list(BS_PRE_CHANNELS),
        "static_channels": list(BS_STATIC_CHANNELS),
        "burn_threshold": float(burn_threshold),
    }
    if extra_meta:
        meta.update(extra_meta)
    with open(os.path.join(weights_dir, "bs_unet_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def load_bs_artifacts(weights_dir, device="cpu"):
    """Loads whatever the training notebook produced, in the layout it writes:

        bs_lightgbm.txt + bs_lightgbm_meta.json   (per-pixel LightGBM, the default model)
        bs_strategy.json                          (strategy + burn_threshold + min_blob_size)
        bs_unet_{i}_best.pt + bs_unet_meta.json   (optional siamese U-Net ensemble)

    Returns a dict ready to splat into `predict_bs_mask`."""
    strategy, burn_threshold, min_blob_size = "lgb", 0.76, 35
    severity_weights, refiner_alpha = (1.0, 1.0, 1.0), 1.0
    strategy_path = os.path.join(weights_dir, "bs_strategy.json")
    if os.path.exists(strategy_path):
        with open(strategy_path, encoding="utf-8") as f:
            sj = json.load(f)
        strategy = sj.get("strategy", strategy)
        burn_threshold = sj.get("burn_threshold", burn_threshold)
        min_blob_size = sj.get("min_blob_size", min_blob_size)
        severity_weights = tuple(sj.get("severity_weights", severity_weights))
        refiner_alpha = sj.get("refiner_alpha", refiner_alpha)

    lgb_model = None
    lgb_path = os.path.join(weights_dir, "bs_lightgbm.txt")
    lgb_meta_path = os.path.join(weights_dir, "bs_lightgbm_meta.json")
    if os.path.exists(lgb_path):
        import lightgbm as lgb  # local import: only needed at inference time

        # NOT lgb.Booster(model_file=...): that opens the file inside LightGBM's
        # native code, which on Windows uses the system codepage and fails on any
        # non-ASCII path ("Could not open ...") even when the file is readable.
        with open(lgb_path, encoding="utf-8") as f:
            lgb_model = lgb.Booster(model_str=f.read())
        if os.path.exists(lgb_meta_path):
            with open(lgb_meta_path, encoding="utf-8") as f:
                # a Booster reloaded from disk has no .best_iteration of its own
                lgb_model.best_iteration = json.load(f)["best_iteration"]

    unet_models = None
    unet_meta_path = os.path.join(weights_dir, "bs_unet_meta.json")
    if strategy in ("unet", "blend") and os.path.exists(unet_meta_path):
        with open(unet_meta_path, encoding="utf-8") as f:
            unet_meta = json.load(f)
        base = unet_meta.get("base", 32)
        unet_models = []
        for i in range(unet_meta["n_models"]):
            m = SiameseUNet(pair_channels=len(BS_PRE_CHANNELS), static_channels=len(BS_STATIC_CHANNELS),
                             n_classes=4, base=base).to(device)
            m.load_state_dict(torch.load(os.path.join(weights_dir, f"bs_unet_{i}_best.pt"), map_location=device))
            m.eval()
            unet_models.append(m)

    refiner_models = None
    refiner_meta_path = os.path.join(weights_dir, "bs_refiner_meta.json")
    if strategy == "refiner" and os.path.exists(refiner_meta_path):
        with open(refiner_meta_path, encoding="utf-8") as f:
            refiner_meta = json.load(f)
        refiner_models = []
        for i in range(refiner_meta["n_models"]):
            m = BSRefiner(in_ch=refiner_meta.get("in_ch", BS_REFINER_IN_CH),
                           n_classes=4, pretrained=False).to(device)
            load_refiner_state(m, os.path.join(weights_dir, f"bs_refiner_{i}_best.pt"),
                                device=device)
            m.eval()
            refiner_models.append(m)

    if strategy == "refiner" and not refiner_models:
        raise FileNotFoundError(
            f"bs_strategy.json просит стратегию 'refiner', но в {weights_dir} нет "
            "bs_refiner_meta.json / bs_refiner_*_best.pt - положите их туда из CHECKPOINT_DIR."
        )
    if lgb_model is None and not unet_models:
        raise FileNotFoundError(
            f"В {weights_dir} нет ни bs_lightgbm.txt, ни весов сиамского U-Net - "
            "положите туда артефакты из CHECKPOINT_DIR после обучения."
        )

    return {
        "lgb_model": lgb_model,
        "unet_models": unet_models,
        "refiner_models": refiner_models,
        "strategy": strategy,
        "burn_threshold": burn_threshold,
        "min_blob_size": min_blob_size,
        "severity_weights": severity_weights,
        "refiner_alpha": refiner_alpha,
    }


## 0. Данные: загрузка с Kaggle

Датасет уже опубликован: https://www.kaggle.com/datasets/shotme/cosmohack2026
(публичный, `shotme/cosmohack2026` уже подставлен ниже) — просто выполняйте
ячейки по порядку, ничего менять не нужно.

Ноутбук рассчитан на запуск в Colab/Kaggle Notebook с GPU. Данные тянутся через
`kagglehub.dataset_download` — ничего вручную скачивать/распаковывать не нужно.
Загрузчик ниже сам определяет, лежит ли `fire-{train,test}-renamed` на Kaggle
как исходный `.tar` или как уже распакованная папка (Data Explorer Kaggle
иногда показывает архивы «развёрнутыми» в дереве) — работает в обоих случаях.

**Тест уже выложен** (`fire-test-renamed`, 180 AF + 89 BS чипов, без масок —
их и предсказываем) и добавлен новой версией того же Kaggle-датасета.
`resolve_test_source(DATASET_DIR)` в ячейках инференса подхватывает его точно
так же, как `resolve_train_source` — train. `test/sample_submission.csv` и
`test/meta.csv` лежат в корне тестового архива (не под `af/`/`bs/`, в отличие
от `train/af/meta.csv` и `train/bs/meta.csv`) — читаются через
`test_source.read_csv("sample_submission.csv")`.

**Авторизация нужна обязательно** — Kaggle API требует ключ даже для скачивания
публичных датасетов (проверено: без него `kagglehub.dataset_download` падает
с ошибкой credentials). Самый чистый вариант под Colab — **Colab Secrets**,
делается один раз на весь Google-аккаунт и работает потом в любом ноутбуке
без единой правки кода:

1. В Colab слева — значок 🔑 (Secrets) → **Add new secret**.
2. Добавить `KAGGLE_USERNAME` = ваш ник на Kaggle.
3. Добавить `KAGGLE_KEY` = API-ключ (kaggle.com → Settings → API →
   **Create New Token** — скачается `kaggle.json`, ключ и username там же).
4. Включить переключатель **Notebook access** у обоих секретов.

После этого ячейка ниже отработает сама. `kagglehub.login()` для Colab **не
использовать** — его виджет не блокирует выполнение ячейки, из-за чего
«Run All» уезжает вперёд к скачиванию раньше, чем токен успевает
сохраниться.


In [ ]:
KAGGLE_DATASET_HANDLE = "shotme/cosmohack2026"
_LOCAL_DEV_PATH = r"..\..\Мониторинг DATA\Мониторинг DATA"  # только для локальной разработки

import os

if os.path.isdir(_LOCAL_DEV_PATH):
    # локальная машина разработчика - без похода в сеть (данные уже лежат на диске)
    DATASET_DIR = _LOCAL_DEV_PATH
    print("Найдены локальные данные - используем их:", DATASET_DIR)
else:
    # Colab / любая другая машина - тянем публичный датасет с Kaggle.
    # Нужны Kaggle-credentials, даже для публичных датасетов (см. markdown выше) -
    # в Colab это делается один раз через Secrets (значок ключа слева), без правок кода.
    import kagglehub
    try:
        DATASET_DIR = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
    except Exception as e:
        raise RuntimeError(
            "Не удалось скачать датасет с Kaggle - похоже, не настроены credentials. "
            "В Colab: значок ключа (Secrets) слева -> Add new secret -> добавить "
            "KAGGLE_USERNAME и KAGGLE_KEY (взять на kaggle.com -> Settings -> API -> "
            "Create New Token) -> включить Notebook access для обоих -> перезапустить "
            "эту ячейку. Подробности - README.md в solution/."
        ) from e
    print("Датасет с Kaggle загружен в:", DATASET_DIR)


def _resolve_source(dataset_dir, kind, source_classes):
    """kind = 'train' или 'test'. fire-{kind}-renamed на Kaggle может быть как исходным
    .tar, так и уже распакованной папкой (зависит от того, разворачивает ли Kaggle архив
    при показе/скачивании) - подхватываем оба варианта без ручных правок."""
    TarChipSource, DirChipSource = source_classes
    tar_path = os.path.join(dataset_dir, f"fire-{kind}-renamed.tar")
    if os.path.isfile(tar_path):
        print(f"Источник {kind}: tar-архив ->", tar_path)
        return TarChipSource(tar_path, root=kind)
    for candidate in (
        os.path.join(dataset_dir, f"fire-{kind}-renamed", kind),
        os.path.join(dataset_dir, kind),
    ):
        if os.path.isdir(candidate):
            print(f"Источник {kind}: распакованная папка ->", candidate)
            return DirChipSource(candidate)
    raise FileNotFoundError(
        f"Не нашли ни {tar_path}, ни распакованную папку {kind}/ внутри {dataset_dir} - "
        "проверьте структуру датасета на Kaggle (Data Explorer)."
    )


def resolve_train_source(dataset_dir):
    return _resolve_source(dataset_dir, "train", (TarChipSource, DirChipSource))


def resolve_test_source(dataset_dir):
    """Тестовые чипы (fire-test-renamed) - без масок, plus test/sample_submission.csv
    и test/meta.csv лежат в корне (не под af/ или bs/, в отличие от train)."""
    return _resolve_source(dataset_dir, "test", (TarChipSource, DirChipSource))


## Важно: где обучать

`RUN_HEAVY_TRAINING` ниже управляет тяжёлыми ячейками (обучение LightGBM /
U-Net). На локальной машине держим его `False` — здесь нет GPU и мало места на
диске. Ставим `True` только на Colab (Runtime → Change runtime type → GPU) или
Kaggle Notebook (Settings → Accelerator → GPU T4x2/P100). Всё остальное
(feature engineering, EDA, быстрые smoke-тесты кода модели) выполняется и
локально.

In [ ]:
import torch

RUN_HEAVY_TRAINING = False  # True только на Colab/Kaggle с GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| RUN_HEAVY_TRAINING:", RUN_HEAVY_TRAINING)


### Куда сохраняются веса

`/content` в Colab — эфемерная файловая система: всё в ней пропадает при
отключении рантайма (по таймауту или вручную), а обучение занимает часы —
терять веса из-за этого обидно. Поэтому чекпоинты и метаданные для
воспроизводимости (гиперпараметры, имена признаков, seed, метрика на
валидации) пишутся на Google Drive, если ноутбук выполняется в Colab, или в
локальную папку `./checkpoints` в остальных случаях (эта машина, Kaggle
Notebook и т.п.).

In [ ]:
import os


def _setup_checkpoint_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        path = "/content/drive/MyDrive/kosmohack2026/checkpoints"
    except ImportError:
        path = os.path.join(".", "checkpoints")
    os.makedirs(path, exist_ok=True)
    return path


CHECKPOINT_DIR = _setup_checkpoint_dir()
print("Чекпоинты сохраняются в:", CHECKPOINT_DIR)


In [ ]:
src = resolve_train_source(DATASET_DIR)
meta_bs = src.meta('bs')
meta_bs.head()

## 1. EDA

- распределение классов степени поражения (`sev1_px`/`sev2_px`/`sev3_px`);
- облачность `cloud_frac`;
- визуальная проверка pre/post RGB + маски.

In [ ]:
sev_cols = ['sev1_px', 'sev2_px', 'sev3_px']
totals = meta_bs[sev_cols].sum()
print(totals)
print('доли:', (totals / totals.sum()).round(3).to_dict())
meta_bs['cloud_frac'].hist(bins=30)
plt.xlabel('cloud_frac'); plt.ylabel('число чипов'); plt.title('Облачность по чипам')

In [ ]:
chip_id = meta_bs.loc[meta_bs['sev2_px'] > 1000, 'chip_id'].iloc[0]
sample = src.load_bs(chip_id)
rgb_pre = np.stack([sample['s2_pre']['B4'], sample['s2_pre']['B3'], sample['s2_pre']['B2']], axis=-1)
rgb_post = np.stack([sample['s2_post']['B4'], sample['s2_post']['B3'], sample['s2_post']['B2']], axis=-1)

def stretch(img, p=2):
    lo, hi = np.percentile(img, p), np.percentile(img, 100 - p)
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(stretch(rgb_pre)); axes[0].set_title('pre (RGB)')
axes[1].imshow(stretch(rgb_post)); axes[1].set_title('post (RGB)')
axes[2].imshow(sample['mask'], cmap='YlOrRd', vmin=0, vmax=3); axes[2].set_title(f'маска, {chip_id}')
for ax in axes: ax.axis('off')
plt.tight_layout()

## 2. Валидационный сплит (по `fire_event_id`, не по пикселям/случайным чипам)

In [ ]:
from sklearn.model_selection import train_test_split

events = meta_bs[['chip_id', 'fire_event_id']].copy()
unique_events = events['fire_event_id'].dropna().unique()
train_events, val_events = train_test_split(unique_events, test_size=0.2, random_state=RANDOM_SEED)
train_ids = events.loc[events['fire_event_id'].isin(train_events), 'chip_id']
val_ids = events.loc[events['fire_event_id'].isin(val_events), 'chip_id']
print(len(train_ids), 'train /', len(val_ids), 'val (уникальных событий:', len(unique_events), ')')

## 3. Пиксельный LightGBM — основная модель BS

**Почему не сеть.** Степень поражения — по сути попиксельная спектральная задача: dNBR, условленный типом покрова (сама постановка говорит, что фиксированные шкалы dNBR не работают, потому что лесу/степи/пашне нужна разная калибровка). Сиамский U-Net видит 179 изображений и голодает по данным, а градиентный бустинг видит **~47 млн размеченных пикселей** из тех же 179 чипов — и это ровно та часть `Score`, где был главный провал (`mIoU_sev`).

Замер на валидации (реальный эксперимент, не предположение):

| модель | IoU_burn | mIoU_sev | 0.35·IoU+0.30·mIoU |
|---|---|---|---|
| baseline (единый порог dNBR) | 0.393 | 0.180 | 0.192 |
| сиамский U-Net (ансамбль из 2 + TTA) | 0.526 | ~0.30 | ~0.274 |
| **пиксельный LightGBM** | **0.546** | **0.556** | **0.358** |

`mIoU_sev` почти удвоился. Плюс инференс ~0.9 с/чип против ~4.4 с/чип у одной сети (а с ансамблем и TTA — в разы дороже), что важно для раздела «Скорость» критериев.

Обучающая таблица — **стратифицированная выборка пикселей**: до `BS_LGB_MAX_PER_CLASS` пикселей каждого класса с каждого чипа. Это и решает дисбаланс (класс 0 занимает ~90% площади), и держит таблицу в разумных размерах по памяти. Из-за сбалансированной выборки вероятности модели откалиброваны под равномерный prior, а не под реальный — поэтому оптимальный порог «есть гарь» подбирается отдельно в секции 5 и обычно оказывается высоким.

In [ ]:
BS_LGB_MAX_PER_CLASS = 5000   # пикселей каждого класса с каждого чипа
BS_LGB_BOUNDARY_EXTRA = 3000  # доп. пиксели у ГРАНИЦЫ гари (см. ниже)
BS_LGB_NUM_LEAVES = 127
BS_LGB_ROUNDS = 700


def build_bs_training_table(source, chip_ids, max_per_class=BS_LGB_MAX_PER_CLASS,
                             boundary_extra=BS_LGB_BOUNDARY_EXTRA,
                             seed=RANDOM_SEED, desc='BS training table'):
    """Стратифицированная выборка пикселей + дополнительная выборка у границы гари.

    Случайные пиксели класса 0 почти все далеко от контура, и модель учится на
    лёгких примерах. Кольцо снаружи гари и полоса внутри - это как раз те пиксели,
    на которых решается IoU_burn (аналог hard negative mining в модуле AF).
    Замер на валидации: 0.3660 -> 0.3737 по вкладу модуля в Score."""
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [], []
    for chip_id in tqdm(list(chip_ids), desc=desc):
        chip = source.load_bs(chip_id)
        feats = build_bs_pixel_features(chip)
        mask = chip['mask']
        valid = nodata_to_ignore(mask, chip['mask_nodata'])
        flat = feats.reshape(feats.shape[0], -1)
        mask_flat, valid_flat = mask.ravel(), valid.ravel()

        for c in range(4):
            idx = np.flatnonzero((mask_flat == c) & valid_flat)
            if len(idx) == 0:
                continue
            take = min(len(idx), max_per_class)
            sel = rng.choice(idx, size=take, replace=False)
            X_parts.append(flat[:, sel].T)
            y_parts.append(np.full(take, c, dtype=np.int32))

        burn = mask > 0
        if boundary_extra and burn.any():
            outer_ring = ndi.binary_dilation(burn, iterations=6) & (~burn)
            inner_band = burn & (~ndi.binary_erosion(burn, iterations=6))
            for zone in (outer_ring, inner_band):
                zone_idx = np.flatnonzero(zone.ravel() & valid_flat)
                if len(zone_idx) == 0:
                    continue
                take = min(len(zone_idx), boundary_extra)
                sel = rng.choice(zone_idx, size=take, replace=False)
                X_parts.append(flat[:, sel].T)
                y_parts.append(mask_flat[sel].astype(np.int32))
    return np.concatenate(X_parts, axis=0), np.concatenate(y_parts, axis=0)


# лёгкая проверка на 2 чипах - фичи строятся, размерности сходятся
_bs_X_smoke, _bs_y_smoke = build_bs_training_table(src, list(train_ids[:2]), max_per_class=500,
                                                     desc='smoke test (2 чипа)')
print('smoke test:', _bs_X_smoke.shape, 'классы:', np.bincount(_bs_y_smoke, minlength=4),
      'NaN:', int(np.isnan(_bs_X_smoke).sum()))

In [ ]:
if RUN_HEAVY_TRAINING:
    import json
    import time

    import lightgbm as lgb

    X_bs_train, y_bs_train = build_bs_training_table(src, train_ids, desc='BS таблица (train)')
    X_bs_val, y_bs_val = build_bs_training_table(src, val_ids, desc='BS таблица (val)')
    print('таблицы:', X_bs_train.shape, X_bs_val.shape,
          '| классы train:', np.bincount(y_bs_train, minlength=4))

    _lc_idx = BS_PIXEL_LANDCOVER_IDX
    _dtrain = lgb.Dataset(X_bs_train, label=y_bs_train, feature_name=BS_PIXEL_FEATURE_NAMES,
                           categorical_feature=[_lc_idx])
    _dval = lgb.Dataset(X_bs_val, label=y_bs_val, feature_name=BS_PIXEL_FEATURE_NAMES,
                         categorical_feature=[_lc_idx], reference=_dtrain)
    bs_lgb_params = dict(
        objective='multiclass', num_class=4, metric='multi_logloss',
        num_leaves=BS_LGB_NUM_LEAVES, learning_rate=0.06,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
        min_data_in_leaf=100, seed=RANDOM_SEED, verbose=-1, num_threads=0,
    )
    bs_lgb_model = lgb.train(
        bs_lgb_params, _dtrain, num_boost_round=BS_LGB_ROUNDS, valid_sets=[_dval],
        callbacks=[lgb.early_stopping(80), lgb.log_evaluation(100)],
    )
    print('best_iteration:', bs_lgb_model.best_iteration)
    _imp = sorted(zip(BS_PIXEL_FEATURE_NAMES, bs_lgb_model.feature_importance()),
                   key=lambda kv: -kv[1])[:15]
    print('топ-15 признаков:', _imp)

    bs_lgb_path = os.path.join(CHECKPOINT_DIR, 'bs_lightgbm.txt')
    bs_lgb_model.save_model(bs_lgb_path)
    with open(os.path.join(CHECKPOINT_DIR, 'bs_lightgbm_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'feature_names': BS_PIXEL_FEATURE_NAMES,
            'categorical_feature_idx': [_lc_idx],
            'params': bs_lgb_params,
            'best_iteration': bs_lgb_model.best_iteration,
            'max_per_class': BS_LGB_MAX_PER_CLASS,
            'boundary_extra': BS_LGB_BOUNDARY_EXTRA,
            'seed': RANDOM_SEED,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, f, ensure_ascii=False, indent=2)
    print('Сохранено:', bs_lgb_path)

    # проверка воспроизводимости: модель с диска даёт те же предсказания
    _reloaded = lgb.Booster(model_file=bs_lgb_path)
    _a = bs_lgb_model.predict(X_bs_val[:200], num_iteration=bs_lgb_model.best_iteration)
    _b = _reloaded.predict(X_bs_val[:200], num_iteration=bs_lgb_model.best_iteration)
    assert np.allclose(_a, _b), 'сохранённая модель не воспроизводит предсказания!'
    print('Проверка воспроизводимости пройдена.')
else:
    print('RUN_HEAVY_TRAINING=False - обучение LightGBM пропущено (Colab/Kaggle).')
    bs_lgb_model = None

## 4. Сиамский U-Net — опционально (`BS_TRAIN_UNET`)

**По умолчанию выключен.** На реальном прогоне ансамбль из двух сиамских U-Net дал `0.35·IoU+0.30·mIoU ≈ 0.274` против `0.358` у пиксельного LightGBM, при этом обучение занимало ~40 минут GPU против ~4 минут CPU, а инференс был в разы дороже. Код оставлен полностью рабочим: если хотите проверить, добавляет ли сеть что-то **в ансамбле** с LightGBM (пространственная связность контура — то, чего попиксельной модели не хватает), поставьте `BS_TRAIN_UNET = True`. Секция 5 честно сравнит три стратегии (LightGBM / U-Net / среднее) на валидации и возьмёт лучшую.

По сравнению с прошлой версией здесь три исправления: `base=16` вместо 32 (в 4 раза меньше параметров — на 179 чипах сеть переобучалась), `crops_per_chip` (на эпоху приходится в 6 раз больше шагов оптимизатора — раньше их было всего ~880 за всё обучение), и **веса классов по корню из обратной частоты** вместо чистой обратной (раньше класс «не горело» получал вес 0.045 против ~1.8 у горелых — сеть была вынуждена предсказывать гарь почти везде, отсюда и порог 0.8).

In [ ]:
BS_TRAIN_UNET = False   # True - обучить сеть и проверить ансамбль с LightGBM
BS_UNET_BASE = 16
BS_UNET_EPOCHS = 20
BS_UNET_CROPS_PER_CHIP = 6
BS_N_MODELS = 1

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class BSChipDataset(Dataset):
    """crops_per_chip > 1 - несколько случайных кропов с одного чипа за эпоху:
    на 179 чипах иначе получается всего ~22 шага оптимизатора на эпоху."""

    def __init__(self, source, chip_ids, augment=False, crop_size=None, crops_per_chip=1):
        self.source = source
        self.chip_ids = list(chip_ids)
        self.augment = augment
        self.crop_size = crop_size
        self.crops_per_chip = crops_per_chip if augment else 1

    def __len__(self):
        return len(self.chip_ids) * self.crops_per_chip

    def __getitem__(self, i):
        chip = self.source.load_bs(self.chip_ids[i % len(self.chip_ids)])
        pre, post, static = build_bs_feature_stack_siamese(chip)
        mask = chip['mask'].astype(np.int64)
        valid = nodata_to_ignore(mask, chip['mask_nodata'])
        mask = np.where(valid, mask, 0)

        if self.crop_size:
            h, w = mask.shape
            cs = self.crop_size
            top = np.random.randint(0, h - cs + 1) if self.augment else (h - cs) // 2
            left = np.random.randint(0, w - cs + 1) if self.augment else (w - cs) // 2
            pre = pre[:, top:top + cs, left:left + cs]
            post = post[:, top:top + cs, left:left + cs]
            static = static[:, top:top + cs, left:left + cs]
            mask = mask[top:top + cs, left:left + cs]
            valid = valid[top:top + cs, left:left + cs]

        if self.augment:
            if np.random.rand() < 0.5:
                pre, post, static = pre[:, :, ::-1].copy(), post[:, :, ::-1].copy(), static[:, :, ::-1].copy()
                mask, valid = mask[:, ::-1].copy(), valid[:, ::-1].copy()
            if np.random.rand() < 0.5:
                pre, post, static = pre[:, ::-1, :].copy(), post[:, ::-1, :].copy(), static[:, ::-1, :].copy()
                mask, valid = mask[::-1, :].copy(), valid[::-1, :].copy()
            k = np.random.randint(0, 4)
            pre = np.ascontiguousarray(np.rot90(pre, k, axes=(1, 2)))
            post = np.ascontiguousarray(np.rot90(post, k, axes=(1, 2)))
            static = np.ascontiguousarray(np.rot90(static, k, axes=(1, 2)))
            mask = np.ascontiguousarray(np.rot90(mask, k))
            valid = np.ascontiguousarray(np.rot90(valid, k))

        return (
            torch.from_numpy(pre.copy()), torch.from_numpy(post.copy()),
            torch.from_numpy(static.copy()), torch.from_numpy(mask.copy()),
            torch.from_numpy(valid.astype(np.float32).copy()),
        )


class DiceCELoss(nn.Module):
    """CE + squared Dice + ordinal-штраф (классы степени упорядочены 0<1<2<3),
    всё - только на valid-пикселях."""

    def __init__(self, class_weights=None, n_classes=4, squared_dice=True, ordinal_weight=0.2):
        super().__init__()
        self.n_classes = n_classes
        self.squared_dice = squared_dice
        self.ordinal_weight = ordinal_weight
        self.ce = nn.CrossEntropyLoss(weight=class_weights, reduction='none')

    def forward(self, logits, target, valid):
        ce = self.ce(logits, target)
        ce = (ce * valid).sum() / valid.sum().clamp_min(1.0)

        probs = F.softmax(logits, dim=1)
        target_onehot = F.one_hot(target, self.n_classes).permute(0, 3, 1, 2).float()
        valid_e = valid.unsqueeze(1)
        inter = (probs * target_onehot * valid_e).sum(dim=(0, 2, 3))
        if self.squared_dice:
            union = ((probs ** 2) * valid_e).sum(dim=(0, 2, 3)) + ((target_onehot ** 2) * valid_e).sum(dim=(0, 2, 3))
        else:
            union = (probs * valid_e).sum(dim=(0, 2, 3)) + (target_onehot * valid_e).sum(dim=(0, 2, 3))
        dice_loss = 1.0 - ((2 * inter + 1.0) / (union + 1.0)).mean()

        class_idx = torch.arange(self.n_classes, device=logits.device, dtype=probs.dtype).view(1, self.n_classes, 1, 1)
        expected_class = (probs * class_idx).sum(dim=1)
        ordinal = ((expected_class - target.float()) ** 2 * valid).sum() / valid.sum().clamp_min(1.0)

        return ce + dice_loss + self.ordinal_weight * ordinal


def compute_bs_class_weights(meta_bs, chip_hw=(512, 512)):
    """Корень из обратной частоты, а не чистая обратная: с чистой класс 0 получал
    вес ~0.045 против ~1.8 у горелых классов, и сеть предсказывала гарь почти везде."""
    total_px = len(meta_bs) * chip_hw[0] * chip_hw[1]
    sev_total = meta_bs[['sev1_px', 'sev2_px', 'sev3_px']].sum()
    counts = np.array([
        total_px - sev_total.sum(), sev_total['sev1_px'], sev_total['sev2_px'], sev_total['sev3_px'],
    ], dtype=np.float64)
    counts = np.clip(counts, 1, None)
    inv = 1.0 / np.sqrt(counts)
    weights = inv / inv.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float32)


def compute_bs_sample_weights(meta_bs, chip_ids, chip_hw=(512, 512)):
    sub = meta_bs.set_index('chip_id').loc[list(chip_ids)]
    severe_frac = (sub['sev2_px'].fillna(0) + sub['sev3_px'].fillna(0)) / (chip_hw[0] * chip_hw[1])
    return torch.tensor((1.0 + 5.0 * severe_frac).values, dtype=torch.double)


# Быстрая проверка модели и лосса (НЕ обучение, ~1 сек на CPU)
_smoke_model = SiameseUNet(pair_channels=len(BS_PRE_CHANNELS), static_channels=len(BS_STATIC_CHANNELS),
                            n_classes=4, base=8)
_logits = _smoke_model(torch.randn(2, len(BS_PRE_CHANNELS), 64, 64),
                        torch.randn(2, len(BS_PRE_CHANNELS), 64, 64),
                        torch.randn(2, len(BS_STATIC_CHANNELS), 64, 64))
_loss = DiceCELoss(n_classes=4)(_logits, torch.randint(0, 4, (2, 64, 64)), torch.ones(2, 64, 64))
_loss.backward()
print('smoke test OK, logits shape:', tuple(_logits.shape), ' loss:', float(_loss.detach()))

In [ ]:
def train_bs_model(source, train_ids, val_ids, meta_bs, model_idx=0, epochs=BS_UNET_EPOCHS,
                    crop_size=256, batch_size=8, lr=1e-3, device=DEVICE, seed=RANDOM_SEED):
    torch.manual_seed(seed)
    train_ds = BSChipDataset(source, train_ids, augment=True, crop_size=crop_size,
                              crops_per_chip=BS_UNET_CROPS_PER_CHIP)
    val_ds = BSChipDataset(source, val_ids, augment=False, crop_size=None)
    sample_weights = compute_bs_sample_weights(meta_bs, train_ids).repeat(BS_UNET_CROPS_PER_CHIP)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)

    model = SiameseUNet(pair_channels=len(BS_PRE_CHANNELS), static_channels=len(BS_STATIC_CHANNELS),
                         n_classes=4, base=BS_UNET_BASE).to(device)
    criterion = DiceCELoss(class_weights=compute_bs_class_weights(meta_bs).to(device), n_classes=4)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_metric, best_state = -1.0, None
    epoch_bar = tqdm(range(epochs), desc=f'U-Net {model_idx}: epochs')
    for epoch in epoch_bar:
        model.train()
        running = 0.0
        batch_bar = tqdm(train_loader, desc=f'epoch {epoch+1}/{epochs} (train)', leave=False)
        for pre, post, static, mask, valid in batch_bar:
            pre, post, static = pre.to(device), post.to(device), static.to(device)
            mask, valid = mask.to(device), valid.to(device)
            optimizer.zero_grad()
            loss = criterion(model(pre, post, static), mask, valid)
            loss.backward()
            optimizer.step()
            running += loss.item() * pre.size(0)
            batch_bar.set_postfix(loss=f'{loss.item():.4f}')
        scheduler.step()

        model.eval()
        acc_binary, acc_sev = ConfusionAccumulator(classes=(1,)), ConfusionAccumulator(classes=(1, 2, 3))
        with torch.no_grad():
            for idx in tqdm(range(len(val_ds)), desc=f'epoch {epoch+1}/{epochs} (val)', leave=False):
                pre, post, static, mask, valid = val_ds[idx]
                pred = model(pre.unsqueeze(0).to(device), post.unsqueeze(0).to(device),
                             static.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
                mask_np, valid_np = mask.numpy(), valid.numpy().astype(bool)
                acc_binary.update(binarize_burn(pred), binarize_burn(mask_np), valid=valid_np)
                acc_sev.update(pred, mask_np, valid=valid_np)
        val_metric = 0.35 * iou_burn(acc_binary) + 0.30 * miou_severity(acc_sev)
        epoch_bar.set_postfix(loss=f'{running/len(train_ds):.4f}',
                               IoU_burn=f'{iou_burn(acc_binary):.3f}',
                               mIoU_sev=f'{miou_severity(acc_sev):.3f}')
        if val_metric > best_metric:
            best_metric = val_metric
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'bs_unet_{model_idx}_best.pt'))
    model.load_state_dict(best_state)
    return model, best_metric


if RUN_HEAVY_TRAINING and BS_TRAIN_UNET:
    import json
    import time

    bs_models = []
    for _mi in range(BS_N_MODELS):
        _m, _vm = train_bs_model(src, train_ids, val_ids, meta_bs, model_idx=_mi, seed=RANDOM_SEED + _mi)
        bs_models.append(_m)
        print(f'U-Net {_mi}: best val_metric={_vm:.4f}')
    with open(os.path.join(CHECKPOINT_DIR, 'bs_unet_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({'n_models': BS_N_MODELS, 'base': BS_UNET_BASE,
                    'pre_channels': list(BS_PRE_CHANNELS),
                    'static_channels': list(BS_STATIC_CHANNELS),
                    'epochs': BS_UNET_EPOCHS, 'seed': RANDOM_SEED,
                    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}, f, ensure_ascii=False, indent=2)
else:
    print('BS_TRAIN_UNET=False (или RUN_HEAVY_TRAINING=False) - сеть не обучается, '
          'работаем на пиксельном LightGBM.')
    bs_models = None

## 5. Выбор стратегии и калибровка порога «есть гарь»

Честно сравниваем на валидации то, что реально доступно: **LightGBM**, **U-Net** (если обучался) и **среднее их вероятностей**. Берём победителя по `0.35·IoU_burn + 0.30·mIoU_sev` — это ровно вклад модуля BS в `Score`.

Порог «есть ли вообще гарь» (`P(1)+P(2)+P(3)`) калибруется **отдельно** от того, какая степень внутри горелой области (там просто `argmax` среди 1/2/3): `IoU_burn` и `mIoU_sev` в `Score` взвешены отдельно, и единый компромиссный порог на всё сразу их обе просаживает. Вероятности по валидации считаются один раз и кэшируются.

In [ ]:
def _bs_probs_for_ids(source, chip_ids, lgb_model=None, unet_models=None, use_tta=False):
    """Считает вероятности (4,H,W) по каждой доступной модели один раз и кэширует
    вместе с маской/valid/landcover - дальше перебор порогов идёт по кэшу."""
    cached = []
    for chip_id in tqdm(list(chip_ids), desc='кэшируем вероятности (val)'):
        chip = source.load_bs(chip_id)
        entry = {
            'mask': chip['mask'],
            'valid': nodata_to_ignore(chip['mask'], chip['mask_nodata']),
            'landcover': chip['aux']['landcover'],
        }
        if lgb_model is not None:
            entry['lgb'] = predict_bs_proba_lgb(chip, lgb_model)
        if unet_models:
            pre, post, static = build_bs_feature_stack_siamese(chip)
            entry['unet'] = predict_bs_proba_tta(
                torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(static),
                unet_models, device=DEVICE, use_tta=use_tta,
            )
        cached.append(entry)
    return cached


def _bs_eval(cached, key_or_blend, threshold):
    acc_b, acc_s = ConfusionAccumulator(classes=(1,)), ConfusionAccumulator(classes=(1, 2, 3))
    for e in cached:
        if key_or_blend == 'blend':
            probs = 0.5 * (e['lgb'] + e['unet'])
        else:
            probs = e[key_or_blend]
        pred = apply_bs_thresholds(probs, burn_threshold=threshold)
        acc_b.update(binarize_burn(pred), binarize_burn(e['mask']), valid=e['valid'])
        acc_s.update(pred, e['mask'], valid=e['valid'])
    ib, ms = iou_burn(acc_b), miou_severity(acc_s)
    return 0.35 * ib + 0.30 * ms, ib, ms


BS_THRESHOLDS = np.linspace(0.30, 0.95, 14)

if RUN_HEAVY_TRAINING and (bs_lgb_model is not None or bs_models):
    _cached = _bs_probs_for_ids(src, val_ids, lgb_model=bs_lgb_model, unet_models=bs_models)

    _strategies = []
    if bs_lgb_model is not None:
        _strategies.append('lgb')
    if bs_models:
        _strategies.append('unet')
    if bs_lgb_model is not None and bs_models:
        _strategies.append('blend')

    _best = None
    for _strategy in _strategies:
        _strategy_best = None
        for _t in tqdm(BS_THRESHOLDS, desc=f'перебор порога ({_strategy})'):
            _s, _ib, _ms = _bs_eval(_cached, _strategy, _t)
            if _strategy_best is None or _s > _strategy_best[0]:
                _strategy_best = (_s, _ib, _ms, float(_t))
            if _best is None or _s > _best[0]:
                _best = (_s, _strategy, float(_t), _ib, _ms)
        print(f'{_strategy:6s}: лучший score={_strategy_best[0]:.4f} '
              f'(IoU_burn={_strategy_best[1]:.4f}, mIoU_sev={_strategy_best[2]:.4f}, '
              f'порог={_strategy_best[3]:.2f})')

    bs_score_val, bs_strategy, burn_threshold, bs_iou_burn_val, bs_miou_sev_val = _best
    print()
    print(f'ВЫБРАНО: стратегия={bs_strategy}, порог={burn_threshold:.3f}')
    print(f'  IoU_burn={bs_iou_burn_val:.4f}  mIoU_sev={bs_miou_sev_val:.4f}  '
          f'вклад в Score (0.35*IoU+0.30*mIoU)={bs_score_val:.4f}')

    _bs_strategy_path = os.path.join(CHECKPOINT_DIR, 'bs_strategy.json')
    with open(_bs_strategy_path, 'w', encoding='utf-8') as f:
        json.dump({'strategy': bs_strategy, 'burn_threshold': burn_threshold,
                    'iou_burn_val': bs_iou_burn_val, 'miou_sev_val': bs_miou_sev_val,
                    'score_val': bs_score_val}, f, ensure_ascii=False, indent=2)
    print('Стратегия и порог сохранены в', _bs_strategy_path)
else:
    bs_strategy, burn_threshold = 'lgb', 0.8

## 6. Постобработка и проверка, помогает ли она

Удаление мелких изолированных блобов (`postprocess_bs_mask` из встроенного `bs_pipeline`) проверяется **на валидации**, а не включается вслепую: на попиксельной модели фильтрация шума обычно помогает `IoU_burn`, но может подрезать мелкие настоящие гари. Дилатация классов 2/3 по умолчанию выключена (она задумывалась под сеть, у которой границы занижены).

In [ ]:
if RUN_HEAVY_TRAINING and (bs_lgb_model is not None or bs_models):
    print('без постобработки: score=%.4f' % bs_score_val)
    _best_pp = (bs_score_val, 0)
    for _mb in [10, 20, 50, 100]:
        _acc_b, _acc_s = ConfusionAccumulator(classes=(1,)), ConfusionAccumulator(classes=(1, 2, 3))
        for _e in _cached:
            _probs = 0.5 * (_e['lgb'] + _e['unet']) if bs_strategy == 'blend' else _e[bs_strategy]
            _pred = apply_bs_thresholds(_probs, burn_threshold=burn_threshold)
            _pred = postprocess_bs_mask(_pred, min_blob_size=_mb, dilate_classes=())
            _acc_b.update(binarize_burn(_pred), binarize_burn(_e['mask']), valid=_e['valid'])
            _acc_s.update(_pred, _e['mask'], valid=_e['valid'])
        _s = 0.35 * iou_burn(_acc_b) + 0.30 * miou_severity(_acc_s)
        print(f'min_blob_size={_mb}: score={_s:.4f} (IoU_burn={iou_burn(_acc_b):.4f}, '
              f'mIoU_sev={miou_severity(_acc_s):.4f})')
        if _s > _best_pp[0]:
            _best_pp = (_s, _mb)
    bs_min_blob_size = _best_pp[1]
    print()
    print('ВЫБРАНО min_blob_size =', bs_min_blob_size, '(0 = постобработка не помогает)')
    with open(os.path.join(CHECKPOINT_DIR, 'bs_strategy.json'), encoding='utf-8') as f:
        _sj = json.load(f)
    _sj['min_blob_size'] = int(bs_min_blob_size)
    _sj['score_val_postproc'] = float(_best_pp[0])
    with open(os.path.join(CHECKPOINT_DIR, 'bs_strategy.json'), 'w', encoding='utf-8') as f:
        json.dump(_sj, f, ensure_ascii=False, indent=2)
else:
    bs_min_blob_size = 0

## 7. Baseline (единый порог dNBR) — контрольная точка

In [ ]:
def baseline_bs_predict(dnbr, thresholds=(0.1, 0.27, 0.66)):
    t0, t1, t2 = thresholds
    out = np.zeros(dnbr.shape, dtype=np.uint8)
    out[(dnbr >= t0) & (dnbr < t1)] = 1
    out[(dnbr >= t1) & (dnbr < t2)] = 2
    out[dnbr >= t2] = 3
    return out


acc_binary_bl, acc_sev_bl = ConfusionAccumulator(classes=(1,)), ConfusionAccumulator(classes=(1, 2, 3))
for chip in tqdm(iter_chips(src, 'bs', val_ids), total=len(val_ids), desc='baseline eval (val)'):
    dnbr_c = _nbr(chip['s2_pre']) - _nbr(chip['s2_post'])
    pred_c = baseline_bs_predict(dnbr_c)
    valid = nodata_to_ignore(chip['mask'], chip['mask_nodata'])
    acc_binary_bl.update(binarize_burn(pred_c), binarize_burn(chip['mask']), valid=valid)
    acc_sev_bl.update(pred_c, chip['mask'], valid=valid)
_bl_score = 0.35 * iou_burn(acc_binary_bl) + 0.30 * miou_severity(acc_sev_bl)
print('baseline: IoU_burn=%.4f  mIoU_sev=%.4f  вклад в Score=%.4f' % (
    iou_burn(acc_binary_bl), miou_severity(acc_sev_bl), _bl_score))

## 8. Быстрый baseline-сабмишн (без обучения, без GPU)

Единый порог dNBR из шага 7 — можно прогнать по тесту за секунды и получить контрольную точку на лидерборде, пока основная модель обучается.

In [ ]:
_bl_test_source = resolve_test_source(DATASET_DIR)
_bl_sample_submission = _bl_test_source.read_csv('sample_submission.csv')
_bl_bs_ids = _bl_sample_submission.loc[
    _bl_sample_submission['chip_id'].str.startswith('BS_'), 'chip_id'
].unique()

_baseline_rows = []
for chip_id in tqdm(_bl_bs_ids, desc='baseline-инференс BS на тесте'):
    chip = _bl_test_source.load_bs(chip_id)
    dnbr_c = _nbr(chip['s2_pre']) - _nbr(chip['s2_post'])
    pred_c = baseline_bs_predict(dnbr_c)
    _baseline_rows.extend(submission_row(chip_id, c, pred_c == c) for c in (1, 2, 3))

submission_bs_baseline = pd.DataFrame(_baseline_rows)
_baseline_path = os.path.join(CHECKPOINT_DIR, 'submission_bs_baseline.csv')
submission_bs_baseline.to_csv(_baseline_path, index=False)
_bl_bs_sample = _bl_sample_submission[_bl_sample_submission['chip_id'].isin(_bl_bs_ids)]
print('Baseline-сабмишн BS сохранён:', _baseline_path, submission_bs_baseline.shape)
print('Проблемы валидации (пусто = ок):', validate_submission(submission_bs_baseline, _bl_bs_sample))

_af_baseline_path = os.path.join(CHECKPOINT_DIR, 'submission_af_baseline.csv')
if os.path.exists(_af_baseline_path):
    _final_baseline = pd.concat([pd.read_csv(_af_baseline_path), submission_bs_baseline], ignore_index=True)
    _final_baseline_path = os.path.join(CHECKPOINT_DIR, 'submission_baseline.csv')
    _final_baseline.to_csv(_final_baseline_path, index=False)
    print('Полный baseline submission.csv собран:', _final_baseline_path, _final_baseline.shape)
    print('Проблемы полной валидации (пусто = ок):',
          validate_submission(_final_baseline, _bl_sample_submission))
else:
    print('submission_af_baseline.csv ещё нет - прогоните секцию baseline-сабмишна ноутбука 1.')
_bl_test_source.close()

## 9. Инференс на тесте + сборка итогового submission.csv

Использует выбранную в секции 5 стратегию и откалиброванные порог/постобработку. Если ничего не обучалось в этой сессии (`RUN_HEAVY_TRAINING=False`) — модель, стратегия и порог подхватываются из `CHECKPOINT_DIR`, так что инференс можно гонять отдельно от обучения. В конце, если `submission_af.csv` из ноутбука 1 уже лежит рядом, обе части собираются в финальный `submission.csv`.

In [ ]:
import json

if bs_lgb_model is None and bs_models is None:
    _lgb_ckpt = os.path.join(CHECKPOINT_DIR, 'bs_lightgbm.txt')
    _lgb_meta = os.path.join(CHECKPOINT_DIR, 'bs_lightgbm_meta.json')
    if os.path.exists(_lgb_ckpt) and os.path.exists(_lgb_meta):
        import lightgbm as lgb
        bs_lgb_model = lgb.Booster(model_file=_lgb_ckpt)
        with open(_lgb_meta, encoding='utf-8') as f:
            bs_lgb_model.best_iteration = json.load(f)['best_iteration']
        print('LightGBM подхвачен из чекпоинта:', _lgb_ckpt)
    _strategy_path = os.path.join(CHECKPOINT_DIR, 'bs_strategy.json')
    if os.path.exists(_strategy_path):
        with open(_strategy_path, encoding='utf-8') as f:
            _sj = json.load(f)
        bs_strategy = _sj.get('strategy', 'lgb')
        burn_threshold = _sj.get('burn_threshold', 0.8)
        bs_min_blob_size = _sj.get('min_blob_size', 0)
        print('Стратегия из чекпоинта:', bs_strategy, '| порог:', burn_threshold,
              '| min_blob_size:', bs_min_blob_size)

if bs_lgb_model is not None or bs_models:
    test_source = resolve_test_source(DATASET_DIR)
    sample_submission = test_source.read_csv('sample_submission.csv')
    bs_test_ids = sample_submission.loc[
        sample_submission['chip_id'].str.startswith('BS_'), 'chip_id'
    ].unique()

    bs_rows = []
    for chip_id in tqdm(bs_test_ids, desc='инференс BS на тесте'):
        chip = test_source.load_bs(chip_id)
        if bs_strategy == 'lgb':
            probs = predict_bs_proba_lgb(chip, bs_lgb_model)
        else:
            pre, post, static = build_bs_feature_stack_siamese(chip)
            probs_unet = predict_bs_proba_tta(
                torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(static),
                bs_models, device=DEVICE, use_tta=True,
            )
            probs = probs_unet if bs_strategy == 'unet' else 0.5 * (
                predict_bs_proba_lgb(chip, bs_lgb_model) + probs_unet)
        pred_mask = apply_bs_thresholds(probs, burn_threshold=burn_threshold)
        if bs_min_blob_size:
            pred_mask = postprocess_bs_mask(pred_mask, min_blob_size=bs_min_blob_size, dilate_classes=())
        bs_rows.extend(submission_row(chip_id, c, pred_mask == c) for c in (1, 2, 3))

    submission_bs = pd.DataFrame(bs_rows)
    bs_sub_path = os.path.join(CHECKPOINT_DIR, 'submission_bs.csv')
    submission_bs.to_csv(bs_sub_path, index=False)
    bs_sample = sample_submission[sample_submission['chip_id'].isin(bs_test_ids)]
    print('Сохранено:', bs_sub_path, submission_bs.shape)
    print('Проблемы валидации (пусто = ок):', validate_submission(submission_bs, bs_sample))

    af_sub_path = os.path.join(CHECKPOINT_DIR, 'submission_af.csv')
    if os.path.exists(af_sub_path):
        final_submission = pd.concat([pd.read_csv(af_sub_path), submission_bs], ignore_index=True)
        final_path = os.path.join(CHECKPOINT_DIR, 'submission.csv')
        final_submission.to_csv(final_path, index=False)
        print('ФИНАЛЬНЫЙ submission.csv собран:', final_path, final_submission.shape)
        print('Проблемы полной валидации (пусто = ок):',
              validate_submission(final_submission, sample_submission))
    else:
        print('submission_af.csv ещё нет в CHECKPOINT_DIR - прогоните ноутбук 1, '
              'затем повторно выполните эту ячейку.')
    test_source.close()
else:
    print('Нет ни обученной модели, ни чекпоинта - инференс пропущен. '
          'Поставьте RUN_HEAVY_TRAINING=True и прогоните секцию 3.')

In [ ]:
src.close()